# Ingestão Streaming Parametrizado

## Objetivo
Sistema de ingestão de dados com suporte a:
1. **Carga Completa (Full Load)**: Carrega dados iniciais de uma tabela base
2. **Carga Incremental (CDC)**: Atualiza a tabela via streaming com dados novos do Kaggle
3. **Parametrização**: Permite criar jobs com 3 execuções independentes (uma para cada tabela)

---

## Arquitetura da Solução

```
Kaggle API → Volume Cache → Landing Zone → Auto Loader (Streaming) → Delta Tables (Bronze)
```

* **Volume Cache**: `/Volumes/workspace/upsell/cdc/kaggle_cache/` - cache temporário do Kaggle
* **Landing Zone**: `/Volumes/workspace/upsell/cdc/kaggle/{tablename}/` - arquivos CSV por tabela
* **Checkpoint**: `/Volumes/workspace/upsell/cdc/checkpoint_{tablename}/` - controle de streaming
* **Target**: `bronze.{tablename}` - tabelas Delta consolidadas

---

## Fluxo de Execução

1. **Cell 4 (Setup)**: Define qual tabela processar (tablename, id_field, timestamp_field)
2. **Cell 6 (Full Load)**: Cria a tabela Delta se não existir (apenas primeira execução)
3. **Cell 8 (Download)**: Baixa dados do Kaggle e organiza por tabela
4. **Cell 9 (Stream Config)**: Configura streaming para a tabela parametrizada
5. **Cell 10 (Stream Start)**: Executa o upsert via streaming
6. **Cell 11 (Validação)**: Compara IDs entre Kaggle e Bronze

---

## Configuração para Jobs Databricks

### Criação de 3 Jobs Independentes:

**Job 1 - Customers:**
```python
tablename = "customers"
id_field = "idCliente"
timestamp_field = "DtAtualizacao"
```

**Job 2 - Transactions:**
```python
tablename = "transactions"
id_field = "idTransacao"
timestamp_field = "DtCriacao"
```

**Job 3 - Transactions Product:**
```python
tablename = "transactions_product"
id_field = "idTransacaoProduto"
timestamp_field = None  # Não tem campo de timestamp

In [0]:
!pip install kaggle
!pip install kagglehub[pandas-datasets]

In [0]:
import kagglehub, shutil, os, delta, time
from kagglehub import KaggleDatasetAdapter
from pyspark import SparkContext
from datetime import datetime, timedelta
from pyspark.sql.functions import col, row_number
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType
from pyspark.sql.window import Window

def table_exists(database, table):
    count = (spark.sql(f"SHOW TABLES FROM {database}")
                  .filter(f"database='{database}' AND tableName='{table}'")
                  .count())    
    return count == 1

In [0]:
# Setup para rodar manual
#tablename = "transactions"
#id_field = "idTransacao"
#timestamp_field = "DtCriacao"
#database = "bronze"

# Setup para rodar os Jobs:
tablename = dbutils.widgets.get("tablename")
id_field = dbutils.widgets.get("id_field")
timestamp_field = dbutils.widgets.get("timestamp_field")
database = dbutils.widgets.get("database")

# Exibindo configuração
print(f"Database: {database}")
print(f"Tabela: {tablename}")
print(f"ID Field: {id_field}")
print(f"Timestamp Field: {timestamp_field}")

In [0]:
tables_config = [
    {
        "database": "bronze",
        "tablename": "customers",
        "id_field": "idCliente",
        "timestamp_field": "DtAtualizacao",
        "schema": StructType([
            StructField("idCliente", StringType(), True),
            StructField("flEmail", StringType(), True),
            StructField("flTwitch", StringType(), True),
            StructField("flYouTube", StringType(), True),
            StructField("flBlueSky", StringType(), True),
            StructField("flInstagram", StringType(), True),
            StructField("qtdePontos", IntegerType(), True),
            StructField("DtCriacao", TimestampType(), True),
            StructField("DtAtualizacao", TimestampType(), True)
        ])
    },
    {
        "database": "bronze",
        "tablename": "transactions",
        "id_field": "idTransacao",
        "timestamp_field": "DtCriacao",
        "schema": StructType([
            StructField("idTransacao", StringType(), True),
            StructField("idCliente", StringType(), True),
            StructField("DtCriacao", TimestampType(), True),
            StructField("QtdePontos", IntegerType(), True),
            StructField("DescSistemaOrigem", StringType(), True),
        ])
    },
    {
        "database": "bronze",
        "tablename": "transactions_product",
        "id_field": "idTransacaoProduto",
        "schema": StructType([
            StructField("idTransacaoProduto", StringType(), True),
            StructField("idTransacao", StringType(), True),
            StructField("IdProduto", StringType(), True),
            StructField("QtdeProduto", IntegerType(), True),
            StructField("vlProduto", DoubleType(), True),
        ])
    }
]

In [0]:
# Busca a configuração da tabela
def get_table_config(tablename):
    for config in tables_config:
        if config["tablename"] == tablename:
            return config
    return None

config = get_table_config(tablename)
if config is None:
    raise ValueError(f"Configuração não encontrada para a tabela: {tablename}")

schema = config["schema"]

if not table_exists(database, tablename):
    print(f"Tabela {database}.{tablename} não existente, criando tabela...")
    df_full = spark.read.format("csv").options(sep=";", header=True).schema(schema).load(f"/Volumes/workspace/upsell/full_load/{tablename}/")
    (df_full.coalesce(1).write.format("delta").mode("overwrite").saveAsTable(f"{database}.{tablename}"))
    print(f"Tabela {database}.{tablename} criada com sucesso")
else:
    print(f"Tabela {database}.{tablename} já existente, ignorando a carga completa")

---
## Atualização via Streaming (CDC)
As próximas cells realizam:
1. Download de dados incrementais do Kaggle
2. Configuração de streams para cada tabela
3. Processamento via `foreachBatch` com lógica de upsert

In [0]:
def ingest_from_kaggle():
    # Definir destino dos arquivos que serão baixados do kaggle
    cache_path = "/Volumes/workspace/upsell/cdc/kaggle_cache"
    os.environ["KAGGLE_CACHE_FOLDER"] = cache_path
    os.makedirs(cache_path, exist_ok=True)

    # Download do dataset direto no cache dentro do Volume
    kaggle_path = kagglehub.dataset_download("teocalvo/teomewhy-loyalty-system")

    arquivos_copiados = []
    # Copiando arquivos para o diretório de destino
    for file in os.listdir(kaggle_path):
        if file.endswith(".csv"):
            base, ext = os.path.splitext(file)
            timestamp = int(time.time())
            new_name = f"{base}_{timestamp}{ext}"

            # Determina o diretório de destino com base no nome do arquivo
            if file.startswith("clientes"):
                target_dir = "/Volumes/workspace/upsell/cdc/kaggle/customers"
            elif file.startswith("transactions_product"):
                target_dir = "/Volumes/workspace/upsell/cdc/kaggle/transactions_product"
            elif file.startswith("transactions"):
                target_dir = "/Volumes/workspace/upsell/cdc/kaggle/transactions"
            else:
                continue

            os.makedirs(target_dir, exist_ok=True)
            full_path = os.path.join(kaggle_path, file)
            dest_file = os.path.join(target_dir, new_name)

            shutil.copy(full_path, dest_file)
            arquivos_copiados.append(dest_file)

    if not arquivos_copiados:
        print("Nenhum arquivo .csv esperado encontrado no dataset")
    else:
        print(f"\n{len(arquivos_copiados)} arquivo(s) prontos nos diretórios correspondentes")

    return [os.path.dirname(f) for f in arquivos_copiados]
        
landing_path = ingest_from_kaggle()

In [0]:
def get_table_config(tablename):
    for config in tables_config:
        if config["tablename"] == tablename:
            return config
    return None

def upsert(df, batch_id, config):
    table = config["tablename"]
    db = config["database"]
    id_col = config["id_field"]
    schema = config["schema"]
    ts_col = config.get("timestamp_field", None)
    delta_table = delta.DeltaTable.forName(spark, f"{db}.{table}")

    # Filtra apenas as colunas do schema esperado
    df_table = df.select([c.name for c in schema.fields if c.name in df.columns])

    # Faz cast dos tipos conforme schema definido
    for field in schema.fields:
        if field.name in df_table.columns:
            df_table = df_table.withColumn(field.name, col(field.name).cast(field.dataType))

    # Se houver campo de timestamp, pega o registro mais recente por id
    if ts_col:
        windowSpec = Window.partitionBy(id_col).orderBy(col(ts_col).desc())
        df_cdc = df_table.withColumn("rn", row_number().over(windowSpec)).filter(col("rn") == 1).drop("rn")
    else:
        df_cdc = df_table

    (delta_table.alias("b")
        .merge(df_cdc.alias("d"), f"b.{id_col} = d.{id_col}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

config = get_table_config(tablename)
if config is None:
    raise ValueError(f"Configuração não encontrada para a tabela: {tablename}")

table = config["tablename"]
schema = config["schema"]

df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("sep", ";")
    .option("header", True)
    .schema(schema)
    .load(f"/Volumes/workspace/upsell/cdc/kaggle/{table}/")
)

stream = (df_stream.writeStream
          .option("checkpointLocation", f"/Volumes/workspace/upsell/cdc/checkpoint_{table}/")
          .foreachBatch(lambda df, batch_id: upsert(df, batch_id, config))
          .trigger(availableNow=True)
)

print(f"Stream configurado para a tabela {table}")

In [0]:
# Inicia o stream e aguarda término
running_stream = stream.start()
running_stream.awaitTermination()
print(f"Stream finalizado para {tablename}")

In [0]:
import os
aggle_path = f"/Volumes/workspace/upsell/cdc/kaggle/{tablename}"
# Verifica se o diretório do Kaggle existe
if not os.path.exists(kaggle_path):
    print(f"Diretório {kaggle_path} não existe")
    print(f"Execute Cell 7 (Carga CDC Do Kaggle) para baixar os dados.")
else:
    # Verifica se a tabela existe no Bronze
    if not table_exists(database, tablename):
        print(f"Tabela {database}.{tablename} não existe.")
        print(f"Execute Cell 5 (Ingestão Carga Completa) primeiro.")
    else:
        df_cdc = spark.read.format("csv").options(sep=";", header=True).load(kaggle_path)
        ids_kaggle = df_cdc.select(id_field).distinct()
        ids_bronze = spark.table(f"{database}.{tablename}").select(id_field).distinct()
        ids_novos = ids_kaggle.subtract(ids_bronze)

        # Exibe resultados
        print(f"\nValidação para tabela: {tablename}")
        print(f"- IDs no Kaggle: {ids_kaggle.count()}")
        print(f"- IDs no Bronze: {ids_bronze.count()}")
        print(f"- Novos IDs (a processar): {ids_novos.count()}")